# Romania 2001: CT, PH, TM, and CJ descriptive placement pilot

This notebook repeats the **frozen Alba-style descriptive definitions** for Constanța, Prahova, Timiș, and Cluj. It uses complete originating-gymnasium applicant reports, complete school-specific admitted/rejected results, program directories, and the newly verified exact-code program pages. It does **not** request webpages.

Run **Cell 1** to see or change the visible settings. Run **Cell 2** to verify source counts before calculating any placement rates. In **Cell 3**, change `RUN_ANALYSIS` to `True` only when you want to make the aggregate CSVs and plots. **Cell 4** displays saved figures; change its two viewing controls without rerunning the analysis.

The peer pool means *observed applicants from the same originating gymnasium*, not all eighth graders. Printed name and admission score are provisional linkage keys because the archived personal identifier is masked. These plots are descriptive associations, not causal estimates.

In [ ]:
# CELL 1 — VISIBLE RESEARCH SETTINGS
# Nothing is downloaded or analyzed when you run this cell. It only locates and
# reloads the companion code so edits made while Cursor was open take effect.
from pathlib import Path
import importlib
import sys
from IPython.display import Image, Markdown, display

def find_analysis_root():
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        candidate = parent / "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis"
        if candidate.is_dir():
            return candidate
        if parent.name == "education_analysis" and (parent / "code").is_dir():
            return parent
    raise FileNotFoundError("Open this notebook from inside the dissertation repository.")

ANALYSIS_ROOT = find_analysis_root()
CODE_DIR = ANALYSIS_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import EDUCATION_20261003_romania_four_county_program_identity_gate as identity_reader
import EDUCATION_20261004_romania_four_county_pilot as shared_plot_code
import EDUCATION_20261009_romania_ct_ph_tm_cj_descriptive_pilot as new_county_pilot
importlib.reload(identity_reader)
importlib.reload(shared_plot_code)
importlib.reload(new_county_pilot)

# Use the four newly reconciled counties, or a subset of them. The combined
# panel pools exactly the counties named here; it is applicant-weighted.
COUNTIES = ("CT", "PH", "TM", "CJ")

# Frozen primary definition: highest fully occupied program cutoff in each
# of six original categories. Top two is a planned, cumulative comparison.
# These refer to CUTOFF TIERS, not simply the two programs with highest scores.
TOP_TIERS = (1, 2)
REQUIRE_FULL_PROGRAMS = True
COMBINE_PROGRAM_CATEGORIES = False
INCLUDE_VOCATIONAL_IN_PRIMARY = True

# Separate denominator comparisons. They never change the program ranking.
COMPARE_EXCLUDING_AMBIGUOUS_EXACT_PROGRAMS = True
COMPARE_EXCLUDING_VOCATIONAL_DESTINATIONS = True
COMPARE_POOLED_SCORE_BANDS = True

# OWN examination-score percentiles: red top 10%, orange 75th–90th,
# gray 50th–75th, blue below 50th. Change to, e.g., (95, 80, 50)
# only if you deliberately want a different descriptive band definition.
AI_PERCENTILE_CUTS = (90, 75, 50)

# Plot resolution only. The CSVs show applicants and distinct gymnasiums
# in every cell so thin or single-school points are visible.
PEER_STRENGTH_BINS = 8
HERO_BINS = 16

settings = new_county_pilot.Settings(
    counties=COUNTIES,
    top_tiers=TOP_TIERS,
    require_full_programs=REQUIRE_FULL_PROGRAMS,
    combine_program_categories=COMBINE_PROGRAM_CATEGORIES,
    include_vocational_in_primary=INCLUDE_VOCATIONAL_IN_PRIMARY,
    compare_excluding_ambiguous=COMPARE_EXCLUDING_AMBIGUOUS_EXACT_PROGRAMS,
    compare_excluding_vocational=COMPARE_EXCLUDING_VOCATIONAL_DESTINATIONS,
    compare_pooled_score_bands=COMPARE_POOLED_SCORE_BANDS,
    ai_percentile_cuts=AI_PERCENTILE_CUTS,
    peer_strength_bins=PEER_STRENGTH_BINS,
    hero_bins=HERO_BINS,
)
settings.validate()
print("Companion code:", new_county_pilot.__file__)
print("Counties:", COUNTIES, "| cutoff tiers:", TOP_TIERS)
print("Full programs:", REQUIRE_FULL_PROGRAMS, "| combine categories:", COMBINE_PROGRAM_CATEGORIES)
print("Own-score bands:", AI_PERCENTILE_CUTS, "| peer bins:", PEER_STRENGTH_BINS)
print("No download or outcome analysis has run.")

## Cell 2 — inspect source counts first

This cell checks the private saved pages and prints **county aggregates only**. The two movement percentages use different denominators: outward placements divide by originating applicants; inward placements divide by destination placements. An away placement is retained when computing gymnasium peers but excluded from the local-program outcome denominator.

In [ ]:
# CELL 2 — OFFLINE SOURCE PREVIEW; NO PLOTS OR DOWNLOADS
# This rereads and verifies the private pages. No personal rows are exported.
source_overview = new_county_pilot.source_preview(settings)
display(source_overview[[
    "county_name", "county", "directory_gymnasiums", "participating_gymnasiums",
    "gymnasiums_with_two_or_more_applicants", "origin_applicants",
    "origin_placed_locally", "outgoing_students", "origin_unassigned",
    "incoming_students", "outgoing_pct_of_origin_applicants",
    "incoming_pct_of_destination_placements",
]].rename(columns={
    "county_name": "County", "county": "Code",
    "directory_gymnasiums": "Gymnasiums in directory",
    "participating_gymnasiums": "Gymnasiums with applicants",
    "gymnasiums_with_two_or_more_applicants": "Gymnasiums with 2+ applicants",
    "origin_applicants": "Applicants", "origin_placed_locally": "Placed in home county",
    "outgoing_students": "Placed in another county", "origin_unassigned": "Unassigned",
    "incoming_students": "Admitted here from elsewhere",
    "outgoing_pct_of_origin_applicants": "Outgoing % of origin applicants",
    "incoming_pct_of_destination_placements": "Incoming % of destination placements",
}).round(2))
print("All four counties passed the saved top-one/top-two program-label source gate.")

## Cell 3 — you start the analysis

Change `RUN_ANALYSIS` below to `True`, then run **this cell**. The companion saves name-free aggregate tables and PNGs in a timestamped output folder. It does not download anything. The full report explains the denominator, movement treatment, band boundaries, and interpretation limits. A repeat run creates a new folder, preserving prior results.

In [ ]:
# CELL 3 — RESEARCHER-CONTROLLED DESCRIPTIVE RUN
RUN_ANALYSIS = False  # Change to True when you want the plots.

if not RUN_ANALYSIS:
    print("Analysis is off. Set RUN_ANALYSIS = True and rerun this cell.")
else:
    run_folder, summary = new_county_pilot.run_pilot(settings, run=True)
    print("Results saved:", run_folder)
    print("Read first:", run_folder / "report.md")
    display(summary[[
        "county", "tier", "source_applicants", "primary_denominator",
        "primary_successes", "primary_observed_success_fraction",
        "qualifying_programs", "qualifying_program_places",
    ]].rename(columns={
        "county": "County", "tier": "Top cutoff tiers",
        "source_applicants": "Applicants", "primary_denominator": "Outcome denominator",
        "primary_successes": "Selected", "primary_observed_success_fraction": "Observed fraction",
        "qualifying_programs": "Winning programs",
        "qualifying_program_places": "Places in winning programs",
    }))

## Cell 4 — view a saved figure

Select `TIER_TO_VIEW` and `FIGURE_TO_VIEW`, then run this cell. `county_bands` shows the four county panels **and** their applicant-weighted pooled panel, with own-score bands as colors. `hero` shows equal-width and equal-number peer bins. `combined_hero` isolates the pooled HERO view. Labels on plotted cells give successful placements/applicants and the number of distinct gymnasiums; the companion CSV contains every cell's support, including low-score bands that are not annotated on the line chart.

In [ ]:
# CELL 4 — DISPLAY SAVED OUTPUTS; NO RECOMPUTATION
TIER_TO_VIEW = 1
FIGURE_TO_VIEW = "county_bands"  # "hero", "combined_hero", "county_bands", "pooled_bands"

if "run_folder" not in globals():
    print("Cell 3 has not produced a run folder in this kernel.")
else:
    display(Markdown((run_folder / "report.md").read_text(encoding="utf-8")))
    filename = {
        "hero": f"hero_top{TIER_TO_VIEW}.png",
        "combined_hero": f"hero_combined_top{TIER_TO_VIEW}.png",
        "county_bands": f"conditional_county_bands_top{TIER_TO_VIEW}.png",
        "pooled_bands": f"conditional_pooled_bands_top{TIER_TO_VIEW}.png",
    }[FIGURE_TO_VIEW]
    figure_path = run_folder / filename
    if not figure_path.exists():
        print("That figure was not requested by the current settings:", figure_path)
    else:
        print("Displaying:", figure_path)
        display(Image(filename=str(figure_path)))